##### Import Libraries

In [3]:
import os
import time
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam, SGD

from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score,precision_score,recall_score,confusion_matrix,classification_report,ConfusionMatrixDisplay)

In [4]:
print("Tensorflow version: ", tf.__version__)
print("Num GPUs Available: ",len(tf.config.list_physical_devices('GPU')))

Tensorflow version:  2.21.0
Num GPUs Available:  0


In [6]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
print("Random seed set to: ",SEED)

Random seed set to:  42


In [9]:
DATASET_DIR=Path("RealWaste")
class_names=sorted([folder.name for folder in DATASET_DIR.iterdir() if folder.is_dir()])
NUM_CLASSES=len(class_names)
print("No. of classes: ",len(class_names))
print("Classes: ")
for i, class_name in enumerate(class_names):
    print(class_name)

No. of classes:  9
Classes: 
Cardboard
Food Organics
Glass
Metal
Miscellaneous Trash
Paper
Plastic
Textile Trash
Vegetation


In [16]:
class_counts = {}

for class_name in class_names:
    class_dir=DATASET_DIR/class_name
    count=sum(1 for file in class_dir.iterdir())
    class_counts[class_name]=count
class_counts_df=pd.DataFrame(list(class_counts.items()),columns=["Class","Number of Images"])
class_counts_df

,Class,Number of Images
0,Cardboard,461
1,Food Organics,411
2,Glass,420
3,Metal,790
4,Miscellaneous Trash,495
5,Paper,500
6,Plastic,921
7,Textile Trash,318
8,Vegetation,436


In [22]:
TOTAL_IMAGES=class_counts_df["Number of Images"].sum()

##### Data Preprocessing

In [25]:
image_paths=[]
labels=[]
for label, class_name in enumerate(class_names):
    class_dir=DATASET_DIR/class_name
    for file in class_dir.iterdir():
        image_paths.append(str(file))
        labels.append(label)

image_paths=np.array(image_paths)
labels=np.array(labels)

print("Total images:",len(image_paths))


Total images: 4752


In [27]:
# data spliting
X_train, X_temp, y_train, y_temp = train_test_split(image_paths,labels,test_size=0.30,stratify=labels,random_state=SEED)
X_val, X_test, y_val, y_test = train_test_split(X_temp,y_temp,test_size=0.50,stratify=y_temp,random_state=SEED)

print("Training images:", len(X_train))
print("Validation images:", len(X_val))
print("Testing images:", len(X_test))

Training images: 3326
Validation images: 713
Testing images: 713


In [28]:
# Tensorflow input
IMG_SIZE=(64,64)
BATCH_SIZE=32

def load_image(path,label):
    image=tf.io.read_file(path)
    image=tf.image.decode_image(image,channels=3,expand_animations=False)
    image=tf.image.resize(image,IMG_SIZE)
    image=tf.cast(image,tf.float32)/255.0
    return image,label

In [30]:
train_ds=tf.data.Dataset.from_tensor_slices((X_train,y_train))
val_ds=tf.data.Dataset.from_tensor_slices((X_val,y_val))
test_ds=tf.data.Dataset.from_tensor_slices((X_test,y_test))

train_ds = train_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
val_ds = val_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)
test_ds = test_ds.map(load_image, num_parallel_calls=tf.data.AUTOTUNE)

train_ds = train_ds.shuffle(len(X_train), seed=SEED).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_ds = val_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

#### Model A - Standard CNN

In [49]:
# Build model A
def build_model_A():
    return models.Sequential([
    layers.Input(shape=(64,64,3)),

    layers.Conv2D(16,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(32,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Conv2D(64,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.Flatten(),

    layers.Dense(64, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

In [50]:
model_A.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 64, 64, 16)     │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 32, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 32, 32, 32)     │         4,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 16, 16, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 4096)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 64)             │       262,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 9)              │           585 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 859,133 (3.28 MB)

 Trainable params: 286,377 (1.09 MB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 572,756 (2.18 MB)

#### Model B - Lightweight CNN

In [51]:
# Build model B
def build_model_B():
    return models.Sequential([
    layers.Input(shape=(64,64,3)),

    layers.SeparableConv2D(16,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.SeparableConv2D(32,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.SeparableConv2D(64,(3,3), padding="same", activation="relu"),
    layers.MaxPooling2D((2,2)),

    layers.GlobalAveragePooling2D(),

    layers.Dense(32, activation="relu"),
    layers.Dense(NUM_CLASSES, activation="softmax")
])

In [52]:
model_B.summary()

Model: "sequential_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ separable_conv2d_6              │ (None, 64, 64, 16)     │            91 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_21 (MaxPooling2D) │ (None, 32, 32, 16)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_7              │ (None, 32, 32, 32)     │           688 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_22 (MaxPooling2D) │ (None, 16, 16, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_8              │ (None, 16, 16, 64)     │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_23 (MaxPooling2D) │ (None, 8, 8, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 9)              │           297 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 5,556 (21.70 KB)

 Trainable params: 5,556 (21.70 KB)

 Non-trainable params: 0 (0.00 B)

####  Optimizer Selection & Tuning

In [53]:
# Compile model A with Adam
model_A=build_model_A()

model_A.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_A_adam = model_A.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

Epoch 1/20


c:\Users\shala\Desktop\Assignment 3 - PR\EN3150_Assignment03_CNN-1\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


104/104 ━━━━━━━━━━━━━━━━━━━━ 11s 46ms/step - accuracy: 0.2865 - loss: 1.9181 - val_accuracy: 0.3801 - val_loss: 1.6182
Epoch 2/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.4233 - loss: 1.5620 - val_accuracy: 0.4446 - val_loss: 1.4634
Epoch 3/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.5045 - loss: 1.3651 - val_accuracy: 0.4811 - val_loss: 1.4243
Epoch 4/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.5463 - loss: 1.2764 - val_accuracy: 0.5638 - val_loss: 1.2569
Epoch 5/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.5926 - loss: 1.1142 - val_accuracy: 0.5835 - val_loss: 1.2314
Epoch 6/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.6401 - loss: 1.0122 - val_accuracy: 0.5568 - val_loss: 1.2617
Epoch 7/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.6627 - loss: 0.9502 - val_accuracy: 0.5905 - val_loss: 1.1227
Epoch 8/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 44ms/step - accuracy: 0.7105 - loss: 0.8216 - val_accuracy: 0.61

In [55]:
# Standard SGD
model_A_sgd=build_model_A()

model_A_sgd.compile(
    optimizer=SGD(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_A_sgd = model_A_sgd.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

Epoch 1/20


c:\Users\shala\Desktop\Assignment 3 - PR\EN3150_Assignment03_CNN-1\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


104/104 ━━━━━━━━━━━━━━━━━━━━ 9s 46ms/step - accuracy: 0.1915 - loss: 2.1427 - val_accuracy: 0.1725 - val_loss: 2.1321
Epoch 2/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - accuracy: 0.1996 - loss: 2.1179 - val_accuracy: 0.1753 - val_loss: 2.1066
Epoch 3/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - accuracy: 0.2189 - loss: 2.0813 - val_accuracy: 0.2412 - val_loss: 2.0576
Epoch 4/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 45ms/step - accuracy: 0.2535 - loss: 2.0225 - val_accuracy: 0.1950 - val_loss: 2.0884
Epoch 5/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 41ms/step - accuracy: 0.2790 - loss: 1.9588 - val_accuracy: 0.3366 - val_loss: 1.9102
Epoch 6/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.3370 - loss: 1.8455 - val_accuracy: 0.3773 - val_loss: 1.7769
Epoch 7/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.3752 - loss: 1.7343 - val_accuracy: 0.3815 - val_loss: 1.6914
Epoch 8/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.4035 - loss: 1.6661 - val_accuracy: 0.415

In [56]:
model_A_momentum=build_model_A()

model_A_momentum.compile(
    optimizer=SGD(learning_rate=0.001,momentum=0.9),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

history_A_momentum = model_A_momentum.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20
)

Epoch 1/20


c:\Users\shala\Desktop\Assignment 3 - PR\EN3150_Assignment03_CNN-1\.venv\Lib\site-packages\keras\src\trainers\epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(


104/104 ━━━━━━━━━━━━━━━━━━━━ 11s 50ms/step - accuracy: 0.1855 - loss: 2.1525 - val_accuracy: 0.1935 - val_loss: 2.1425
Epoch 2/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.1984 - loss: 2.1323 - val_accuracy: 0.1935 - val_loss: 2.1273
Epoch 3/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 43ms/step - accuracy: 0.2066 - loss: 2.1151 - val_accuracy: 0.2090 - val_loss: 2.0995
Epoch 4/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 43ms/step - accuracy: 0.2288 - loss: 2.0774 - val_accuracy: 0.2356 - val_loss: 2.0604
Epoch 5/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 42ms/step - accuracy: 0.2643 - loss: 2.0231 - val_accuracy: 0.2665 - val_loss: 1.9885
Epoch 6/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 8s 44ms/step - accuracy: 0.2949 - loss: 1.9495 - val_accuracy: 0.2945 - val_loss: 1.8993
Epoch 7/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 41ms/step - accuracy: 0.3382 - loss: 1.8734 - val_accuracy: 0.3661 - val_loss: 1.8163
Epoch 8/20
104/104 ━━━━━━━━━━━━━━━━━━━━ 7s 40ms/step - accuracy: 0.3620 - loss: 1.8105 - val_accuracy: 0.37

In [57]:
print("Adam:")
print("Final validation accuracy:", history_A_adam.history["val_accuracy"][-1])
print("Final validation loss:", history_A_adam.history["val_loss"][-1])

print("\nSGD:")
print("Final validation accuracy:", history_A_sgd.history["val_accuracy"][-1])
print("Final validation loss:", history_A_sgd.history["val_loss"][-1])

print("\nSGD + Momentum:")
print("Final validation accuracy:", history_A_momentum.history["val_accuracy"][-1])
print("Final validation loss:", history_A_momentum.history["val_loss"][-1])

Adam:
Final validation accuracy: 0.6395511627197266
Final validation loss: 1.5193792581558228

SGD:
Final validation accuracy: 0.5413744449615479
Final validation loss: 1.3337031602859497

SGD + Momentum:
Final validation accuracy: 0.5483871102333069
Final validation loss: 1.2942930459976196


#### Custom Model Training & Evaluation